# TCC-04 — Sprint 2: Setup do ambiente e treinamento do modelo base (v2)

**Projeto:** Tonare — classificador de melanoma (mel vs nv) com fairness por tom de pele
**Framework:** PyTorch | **Treino real:** Kaggle Notebooks (GPU) | **Local (CPU):** só smoke test

O notebook detecta o ambiente sozinho (`IS_KAGGLE`): local roda um smoke test pequeno,
no Kaggle roda o treino de verdade.

## Por que a v2

A v1 (ResNet50, 5+8 épocas, 5 cópias fixas de cada melanoma vindas da Sprint 1) teve
**recall de mel = 0,58 no teste**, com AUC de validação 0,89. Diagnóstico: recall de treino
0,93 → 0,999 enquanto o de validação ficou em ~0,5, e a loss de validação foi mínima na época 2
e depois só subiu. Ou seja, **overfitting**: o treino tem só 428 lesões distintas de melanoma
(771 imagens) e o modelo decorou as cópias fixas. Mudanças da v2:

1. Treino só com imagens **originais** + augmentation **aleatório a cada época** (rotação livre,
   flips, recorte, brilho/contraste leves; sem mexer em matiz/saturação — cor é sinal diagnóstico).
2. Desbalanceamento (1:6) tratado só pela **loss ponderada** (pesos "balanced" calculados do treino).
3. **Early stopping** e seleção do melhor modelo pela **AUC de validação** (não pela última época).
4. Escolha do **limiar de decisão** na validação, com meta de sensibilidade para melanoma (o teste
   só é olhado no final, uma vez).

### Como rodar no Kaggle
Anexe o dataset `tonare-tcc1` (pasta `processed/`), ligue GPU + Internet, e na Célula 1 cole o
conteúdo de `scripts/torch_dataset.py` no lugar do import (import relativo não funciona lá).


In [1]:
import os
import sys
from pathlib import Path

# Local: importa do repositório. No Kaggle: cole aqui o conteúdo de scripts/torch_dataset.py
if not os.path.exists("/kaggle/input"):
    sys.path.append(str(Path("..") / "scripts"))
    from torch_dataset import CLASS_NAMES, LABEL_MAP, build_dataloaders, build_datasets


In [2]:
IS_KAGGLE = os.path.exists("/kaggle/input")

SMOKE_TEST = not IS_KAGGLE      # local (CPU) = smoke test; Kaggle (GPU) = treino real
ARCHITECTURE = "resnet50"       # ou "efficientnet_b0"
SEED = 42

EPOCHS_STAGE1 = 1 if SMOKE_TEST else 5    # máximo; early stopping pode parar antes
EPOCHS_STAGE2 = 1 if SMOKE_TEST else 12   # máximo; early stopping pode parar antes
PATIENCE = 3                              # épocas sem melhora de AUC antes de parar
BATCH_SIZE = 4 if SMOKE_TEST else 32
LIMIT_PER_SPLIT = 40 if SMOKE_TEST else None
NUM_WORKERS = 0 if SMOKE_TEST else 4

MEL_EXTRA_MULTIPLIER = 1.0   # peso extra pra mel ALÉM do balanceamento (mexer só depois de ver resultados)
TARGET_RECALL_MEL = 0.90     # meta clínica de sensibilidade, usada só pra escolher o limiar na VALIDAÇÃO

print(f"IS_KAGGLE={IS_KAGGLE} | SMOKE_TEST={SMOKE_TEST} | arquitetura={ARCHITECTURE} | batch_size={BATCH_SIZE}")


IS_KAGGLE=False | SMOKE_TEST=True | arquitetura=resnet50 | batch_size=4


In [3]:
import copy
import random

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
from sklearn.utils.class_weight import compute_class_weight
from torch import optim
from torchvision import models

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")


Device: cpu


## 1. Dados e dataloader

No Kaggle o dataset chega sem a pasta `data/` por cima (`.../processed/...`), então criamos um
link simbólico em `/kaggle/working/data/processed` para o código achar `data/processed/...`.

In [4]:
if IS_KAGGLE:
    KAGGLE_PROCESSED = "/kaggle/input/datasets/sofyaoliveiras/tonare-tcc1/processed"
    os.makedirs("/kaggle/working/data", exist_ok=True)
    link = "/kaggle/working/data/processed"
    if os.path.islink(link) or os.path.exists(link):
        os.remove(link)
    os.symlink(KAGGLE_PROCESSED, link)
    BASE_DIR = Path("/kaggle/working")
    OUT_DIR = Path("/kaggle/working")
else:
    BASE_DIR = Path("..")
    OUT_DIR = BASE_DIR / "data" / "processed"

print("BASE_DIR:", BASE_DIR, "| OUT_DIR:", OUT_DIR)


BASE_DIR: .. | OUT_DIR: ..\data\processed


In [5]:
datasets = build_datasets(base_dir=BASE_DIR, limit_per_split=LIMIT_PER_SPLIT,
                          use_precomputed_aug=False, strong_aug=True)
loaders = build_dataloaders(datasets, batch_size=BATCH_SIZE, num_workers=NUM_WORKERS)

for split, ds in datasets.items():
    contagem = ds.df["dx"].value_counts().to_dict()
    print(f"{split}: {len(ds)} imagens {contagem}")


train: 40 imagens {'mel': 20, 'nv': 20}
val: 40 imagens {'mel': 20, 'nv': 20}
test: 40 imagens {'mel': 20, 'nv': 20}


In [6]:
imgs, labels = next(iter(loaders["train"]))
print("Shape do batch de imagens:", imgs.shape)
print("Labels do batch:", labels.tolist())
print("Range de valores:", float(imgs.min()), "a", float(imgs.max()))

assert imgs.shape[1:] == (3, 224, 224), "dataloader não está entregando imagens 224x224x3"
print("\nDataloader validado: uma iteração de teste passou sem erro.")


Shape do batch de imagens: torch.Size([4, 3, 224, 224])
Labels do batch: [1, 1, 1, 1]
Range de valores: -2.0665297508239746 a 2.6051416397094727

Dataloader validado: uma iteração de teste passou sem erro.


## 2. Modelo pré-treinado

In [7]:
def build_model(architecture: str, num_classes: int = 2, pretrained: bool = True) -> nn.Module:
    if architecture == "resnet50":
        weights = models.ResNet50_Weights.IMAGENET1K_V2 if pretrained else None
        model = models.resnet50(weights=weights)
        model.fc = nn.Linear(model.fc.in_features, num_classes)
    elif architecture == "efficientnet_b0":
        weights = models.EfficientNet_B0_Weights.IMAGENET1K_V1 if pretrained else None
        model = models.efficientnet_b0(weights=weights)
        model.classifier[1] = nn.Linear(model.classifier[1].in_features, num_classes)
    else:
        raise ValueError(f"Arquitetura não suportada: {architecture}")
    return model


def set_backbone_trainable(model: nn.Module, architecture: str, trainable: bool) -> None:
    # Congela/descongela tudo, MENOS a camada final de classificação.
    for param in model.parameters():
        param.requires_grad = trainable
    head = model.fc if architecture == "resnet50" else model.classifier
    for param in head.parameters():
        param.requires_grad = True


model = build_model(ARCHITECTURE).to(DEVICE)
set_backbone_trainable(model, ARCHITECTURE, trainable=False)

n_treinaveis = sum(p.numel() for p in model.parameters() if p.requires_grad)
n_total = sum(p.numel() for p in model.parameters())
print(f"Parâmetros treináveis (Estágio 1): {n_treinaveis:,} de {n_total:,} total")


Parâmetros treináveis (Estágio 1): 4,098 de 23,512,130 total


## 3. Loss ponderada

O treino agora tem só imagens originais (~1:6 mel:nv), então o peso "balanced" já dá ~6x mais
importância ao erro em mel. `MEL_EXTRA_MULTIPLIER` (default 1.0) permite reforçar ainda mais.

In [8]:
train_labels = datasets["train"].df["dx"].map(LABEL_MAP).values
base_weights = compute_class_weight(class_weight="balanced", classes=np.array([0, 1]), y=train_labels)
class_weights = base_weights.copy()
class_weights[1] *= MEL_EXTRA_MULTIPLIER
criterion = nn.CrossEntropyLoss(weight=torch.tensor(class_weights, dtype=torch.float32).to(DEVICE))

print(f"Distribuição do treino: {pd.Series(train_labels).value_counts().to_dict()} (0=nv, 1=mel)")
print(f"Pesos da loss: nv={class_weights[0]:.3f}, mel={class_weights[1]:.3f}")
if not SMOKE_TEST:  # o smoke test é balanceado por construção (20 mel/20 nv)
    assert class_weights[1] > class_weights[0], "peso de mel deveria ser maior que o de nv"


Distribuição do treino: {1: 20, 0: 20} (0=nv, 1=mel)
Pesos da loss: nv=1.000, mel=1.000


## 4. Loop de treino (early stopping pela AUC de validação)

In [9]:
def run_epoch(model, loader, criterion, optimizer=None) -> dict:
    is_train = optimizer is not None
    model.train() if is_train else model.eval()

    total_loss, all_probs, all_labels = 0.0, [], []
    with torch.set_grad_enabled(is_train):
        for imgs, labels in loader:
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
            outputs = model(imgs)
            loss = criterion(outputs, labels)

            if is_train:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

            total_loss += loss.item() * imgs.size(0)
            all_probs.extend(F.softmax(outputs.detach(), dim=1)[:, 1].cpu().tolist())
            all_labels.extend(labels.cpu().tolist())

    labels_arr, probs_arr = np.array(all_labels), np.array(all_probs)
    preds = (probs_arr >= 0.5).astype(int)
    report = classification_report(labels_arr, preds, target_names=CLASS_NAMES,
                                    output_dict=True, zero_division=0)
    auc = roc_auc_score(labels_arr, probs_arr) if len(set(all_labels)) > 1 else float("nan")
    return {"loss": total_loss / len(loader.dataset), "accuracy": report["accuracy"],
            "recall_mel": report["mel"]["recall"], "auc": auc,
            "labels": labels_arr, "probs": probs_arr}


def train_stage(model, loaders, criterion, optimizer, n_epochs: int, stage_name: str, patience: int = PATIENCE):
    history, best_auc, best_state, sem_melhora = [], -1.0, None, 0
    print(f"[{stage_name}] pesos da loss (nv, mel): {[round(w, 3) for w in criterion.weight.tolist()]}")
    for epoch in range(1, n_epochs + 1):
        tr = run_epoch(model, loaders["train"], criterion, optimizer)
        va = run_epoch(model, loaders["val"], criterion, optimizer=None)
        print(f"[{stage_name}] época {epoch}/{n_epochs} | "
              f"treino: loss={tr['loss']:.3f} auc={tr['auc']:.3f} | "
              f"val: loss={va['loss']:.3f} auc={va['auc']:.3f} recall_mel@0.5={va['recall_mel']:.3f}")
        history.append({"epoch": epoch, "train": tr, "val": va})

        if va["auc"] > best_auc:
            best_auc, best_state, sem_melhora = va["auc"], copy.deepcopy(model.state_dict()), 0
        else:
            sem_melhora += 1
            if sem_melhora >= patience:
                print(f"[{stage_name}] early stopping: sem melhora da AUC de validação há {patience} épocas")
                break

    if best_state is not None:
        model.load_state_dict(best_state)
        print(f"[{stage_name}] modelo restaurado para a melhor época (val AUC = {best_auc:.3f})")
    return history


## 5. Estágio 1 — só a camada final
Backbone congelado, LR alto (1e-3) porque são poucos parâmetros novos.

In [10]:
optimizer_stage1 = optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-3, weight_decay=1e-2)
history_stage1 = train_stage(model, loaders, criterion, optimizer_stage1, EPOCHS_STAGE1, "Estágio 1")


[Estágio 1] pesos da loss (nv, mel): [1.0, 1.0]


[Estágio 1] época 1/1 | treino: loss=0.714 auc=0.545 | val: loss=0.663 auc=0.723 recall_mel@0.5=0.400
[Estágio 1] modelo restaurado para a melhor época (val AUC = 0.723)


## 6. Estágio 2 — fine-tuning profundo
Backbone descongelado, LR bem baixo (1e-5) + weight decay para regularizar.

In [11]:
set_backbone_trainable(model, ARCHITECTURE, trainable=True)
n_treinaveis = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Parâmetros treináveis (Estágio 2): {n_treinaveis:,}")

optimizer_stage2 = optim.AdamW(model.parameters(), lr=1e-5, weight_decay=1e-2)
history_stage2 = train_stage(model, loaders, criterion, optimizer_stage2, EPOCHS_STAGE2, "Estágio 2")


Parâmetros treináveis (Estágio 2): 23,512,130
[Estágio 2] pesos da loss (nv, mel): [1.0, 1.0]


[Estágio 2] época 1/1 | treino: loss=0.612 auc=0.885 | val: loss=0.628 auc=0.863 recall_mel@0.5=0.600
[Estágio 2] modelo restaurado para a melhor época (val AUC = 0.863)


## 7. Escolha do limiar (só na validação)

O modelo dá uma probabilidade de melanoma; o limiar decide a partir de quanto marcamos como `mel`.
Baixar o limiar aumenta a sensibilidade (menos melanomas perdidos) e custa especificidade
(mais nevos benignos marcados). A tabela mostra esse custo.

In [12]:
def sens_spec_prec(labels, probs, threshold):
    preds = (probs >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(labels, preds, labels=[0, 1]).ravel()
    sens = tp / (tp + fn) if (tp + fn) else float("nan")
    spec = tn / (tn + fp) if (tn + fp) else float("nan")
    prec = tp / (tp + fp) if (tp + fp) else float("nan")
    return sens, spec, prec


val_res = run_epoch(model, loaders["val"], criterion)
val_labels, val_probs = val_res["labels"], val_res["probs"]
print(f"Validação: AUC-ROC = {val_res['auc']:.3f}\n")

print(f"{'limiar':>7} {'sensibilidade(mel)':>19} {'especificidade':>15} {'precisão(mel)':>14}")
for t in np.arange(0.05, 0.96, 0.05):
    sens, spec, prec = sens_spec_prec(val_labels, val_probs, t)
    print(f"{t:>7.2f} {sens:>19.3f} {spec:>15.3f} {prec:>14.3f}")


Validação: AUC-ROC = 0.863

 limiar  sensibilidade(mel)  especificidade  precisão(mel)
   0.05               1.000           0.000          0.500
   0.10               1.000           0.000          0.500
   0.15               1.000           0.000          0.500
   0.20               1.000           0.000          0.500
   0.25               1.000           0.000          0.500
   0.30               1.000           0.000          0.500
   0.35               1.000           0.000          0.500
   0.40               1.000           0.250          0.571
   0.45               1.000           0.750          0.800
   0.50               0.600           0.850          0.800
   0.55               0.100           0.950          0.667
   0.60               0.000           1.000            nan
   0.65               0.000           1.000            nan
   0.70               0.000           1.000            nan
   0.75               0.000           1.000            nan
   0.80               0.000 

## 8. Resultado final no teste (olhado uma única vez)

Limiar escolhido na **validação**: o maior limiar que ainda atinge `TARGET_RECALL_MEL`
(maior limiar = menos falsos positivos dentro da meta). Se nenhum atinge, usa o de maior
sensibilidade e avisa.

In [13]:
thresholds = np.round(np.arange(0.01, 0.99, 0.01), 2)
sens_por_t = {t: sens_spec_prec(val_labels, val_probs, t)[0] for t in thresholds}
candidatos = [t for t, s in sens_por_t.items() if s >= TARGET_RECALL_MEL]

if candidatos:
    best_threshold = max(candidatos)
    print(f"Limiar escolhido na validação: {best_threshold:.2f} (sensibilidade >= {TARGET_RECALL_MEL})")
else:
    best_threshold = max(sens_por_t, key=lambda t: (sens_por_t[t], t))  # maior sensibilidade; empate -> maior limiar
    print(f"AVISO: nenhum limiar atinge sensibilidade {TARGET_RECALL_MEL} na validação. "
          f"Usando o de maior sensibilidade: {best_threshold:.2f} ({sens_por_t[best_threshold]:.3f})")

test_res = run_epoch(model, loaders["test"], criterion)
test_labels, test_probs = test_res["labels"], test_res["probs"]
print(f"\nTESTE — AUC-ROC = {test_res['auc']:.3f}")
for nome, t in [("limiar padrão 0.50", 0.5), (f"limiar escolhido {best_threshold:.2f}", best_threshold)]:
    sens, spec, prec = sens_spec_prec(test_labels, test_probs, t)
    tn, fp, fn, tp = confusion_matrix(test_labels, (test_probs >= t).astype(int), labels=[0, 1]).ravel()
    print(f"  {nome}: sensibilidade(mel)={sens:.3f} especificidade={spec:.3f} precisão(mel)={prec:.3f} "
          f"| melanomas perdidos: {fn} de {tp + fn} | nevos marcados como mel: {fp} de {tn + fp}")


Limiar escolhido na validação: 0.47 (sensibilidade >= 0.9)



TESTE — AUC-ROC = 0.835
  limiar padrão 0.50: sensibilidade(mel)=0.250 especificidade=0.850 precisão(mel)=0.625 | melanomas perdidos: 15 de 20 | nevos marcados como mel: 3 de 20
  limiar escolhido 0.47: sensibilidade(mel)=0.700 especificidade=0.800 precisão(mel)=0.778 | melanomas perdidos: 6 de 20 | nevos marcados como mel: 4 de 20


In [14]:
checkpoint_path = OUT_DIR / f"model_{ARCHITECTURE}_baseline.pt"
torch.save({
    "model_state_dict": model.state_dict(),
    "architecture": ARCHITECTURE,
    "label_map": LABEL_MAP,
    "threshold": float(best_threshold),
    "target_recall_mel": TARGET_RECALL_MEL,
    "mel_extra_multiplier": MEL_EXTRA_MULTIPLIER,
    "val_auc": float(val_res["auc"]),
    "smoke_test": SMOKE_TEST,
}, checkpoint_path)
print(f"Checkpoint salvo em: {checkpoint_path}")
if SMOKE_TEST:
    print("SMOKE TEST (CPU, amostra pequena): serve só para provar que o pipeline roda. "
          "Rode no Kaggle (GPU) para o treino de verdade.")


Checkpoint salvo em: ..\data\processed\model_resnet50_baseline.pt
SMOKE TEST (CPU, amostra pequena): serve só para provar que o pipeline roda. Rode no Kaggle (GPU) para o treino de verdade.


## Próximo passo (Sprint 3)
Carregar o checkpoint (com o `threshold` salvo nele) e avaliar sobre o **DDI**
(`data/processed/ddi/ddi_manifest.csv`), comparando sensibilidade/especificidade entre as 3 faixas
de tom de pele — o resultado central de fairness do TCC.